### Frontier Transformation Engineer finder
This notebook takes the Trainings report from Partner Center Insights, and filters certifications for learners working toward the Frontier Transformation Engineer Badge.

In [17]:
import pandas as pd

filename = 'SampleTrainings.csv'  # Update with your actual filename

if filename.lower().endswith('.tsv'):
    df = pd.read_csv(filename, sep='\t')
elif filename.lower().endswith('.csv'):
    df = pd.read_csv(filename)
else:
    raise ValueError("Unsupported file type. Use a .csv or .tsv file.")

print(f"Loaded {len(df):,} training activities")
unique_learners = df['AADUserId'].nunique()
print(f"Loaded {unique_learners:,} unique learners")


Loaded 40,000 training activities
Loaded 10,178 unique learners


### Learners with all three certification requirements
Learner has all the following active certifications:
- GitHub Copilot
**AND**
- Microsoft Certified: Agentic AI Business Solutions Architect

and **EITHER**
- Microsoft Certified: Azure AI Engineer Associate **OR**
- Microsoft Certified: Azure AI Apps and Agents Developer Associate'

In [18]:
# Define Azure AI certification group (old and new)
azure_ai_certs = [
    'Microsoft Certified: Azure AI Engineer Associate',
    'Microsoft Certified: Azure AI Apps and Agents Developer Associate'
    # Add more aliases if needed in future
 ]
target_certs = [
    'Azure AI',  # Placeholder for either of the above
    'GitHub Copilot',
    'Microsoft Certified: Agentic AI Business Solutions Architect'
 ]

# Filter for target certifications that are Active and of type Certification, with valid Email
filtered = df[
    ((
        (df['TrainingTitle'].isin(azure_ai_certs)) |
        (df['TrainingTitle'] == 'GitHub Copilot') |
        (df['TrainingTitle'] == 'Microsoft Certified: Agentic AI Business Solutions Architect')
    )) &
    (df['ActivationStatus'] == 'Active') &
    (df['TrainingType'] == 'Certification') &
    (df['Email'].notna()) &
    (df['Email'].str.strip() != '')
 ]

# Map TrainingTitle to logical cert bucket for grouping
def logical_cert(row):
    if row['TrainingTitle'] in azure_ai_certs:
        return 'Azure AI'
    return row['TrainingTitle']

filtered = filtered.copy()
filtered['LogicalCert'] = filtered.apply(logical_cert, axis=1)

# Find learners who have earned all 3 logical certifications
cert_counts = filtered.groupby('Email')['LogicalCert'].nunique()
qualified_ids = cert_counts[cert_counts == 3].index

# Build output table with one row per learner
qualified = filtered[filtered['Email'].isin(qualified_ids)]
result = (
    qualified.groupby('Email')
    .agg(
        FirstName=('IndividualFirstName', 'first'),
        LastName=('IndividualLastName', 'first'),
        CorpEmail=('CorpEmail', 'first')
    )
    .reset_index()
 )
result['Name'] = result['FirstName'].str.cat(result['LastName'], sep=' ').str.strip()
result = result[['Name', 'CorpEmail']]

print(f'Learners with all 3 certifications (Active): {len(result)}')
result

Learners with all 3 certifications (Active): 5


,Name,CorpEmail
0,Allan Kelly,allan.kelly@contoso.com
1,Damian Ward,damian.ward@contoso.com
2,Hilary Evans,hilary.evans@contoso.com
3,Martin Davies,martin.davies@contoso.com
4,Stuart Parsons,stuart.parsons@contoso.com


### Learners with exactly 2 out of 3 certification requirements

In [19]:
# Use the logical_cert mapping and updated logic for 2 out of 3
two_of_three_ids = cert_counts[cert_counts == 2].index
two_of_three = filtered[filtered['Email'].isin(two_of_three_ids)]

# For each learner, find the missing logical certification
rows = []
for email, group in two_of_three.groupby('Email'):
    earned = set(group['LogicalCert'])
    missing = set(target_certs) - earned
    row = group.iloc[0]
    name = f"{row['IndividualFirstName']} {row['IndividualLastName']}".strip()
    for cert in missing:
        # If Azure AI is missing, always recommend the new cert only
        if cert == 'Azure AI':
            rows.append({'Name': name, 'CorpEmail': row['CorpEmail'], 'Missing Certification': 'Microsoft Certified: Azure AI Apps and Agents Developer Associate'})
        else:
            rows.append({'Name': name, 'CorpEmail': row['CorpEmail'], 'Missing Certification': cert})

result_2of3 = pd.DataFrame(rows)

print(f'Learners with 2 out of 3 certifications (Active): {result_2of3["CorpEmail"].nunique()}')
result_2of3

Learners with 2 out of 3 certifications (Active): 100


,Name,CorpEmail,Missing Certification
0,Abdul Brown,abdul.brown@contoso.com,Microsoft Certified: Agentic AI Business Solut...
1,Alexander Peters,alexander.peters@contoso.com,GitHub Copilot
2,Alexander Shepherd,alexander.shepherd@contoso.com,Microsoft Certified: Azure AI Apps and Agents ...
3,Amber Preston,amber.preston@contoso.com,Microsoft Certified: Azure AI Apps and Agents ...
4,Andrea Adams,andrea.adams@contoso.com,Microsoft Certified: Agentic AI Business Solut...
...,...,...,...
95,Tracy Lloyd,tracy.lloyd@contoso.com,Microsoft Certified: Azure AI Apps and Agents ...
96,Trevor Ingram,trevor.ingram@contoso.com,GitHub Copilot
97,Victor Singh,victor.singh@contoso.com,Microsoft Certified: Azure AI Apps and Agents ...
98,Yvonne Page,yvonne.page@contoso.com,Microsoft Certified: Agentic AI Business Solut...


### At risk certifications
All target certifications, in order of expiration date, earliest to latest

In [20]:
# At risk certifications
at_risk = filtered.copy()
at_risk['ExpirationDate'] = pd.to_datetime(at_risk['ExpirationDate'])
at_risk = at_risk.dropna(subset=['ExpirationDate'])
at_risk = at_risk.sort_values('ExpirationDate')

at_risk['Name'] = at_risk['IndividualFirstName'].str.cat(at_risk['IndividualLastName'], sep=' ').str.strip()
at_risk = at_risk[['Name', 'CorpEmail', 'TrainingTitle', 'ExpirationDate']].rename(
    columns={'TrainingTitle': 'Certification', 'ExpirationDate': 'Expiration Date'}
)
at_risk = at_risk.reset_index(drop=True)

today = pd.Timestamp.today().normalize()
cutoff = today + pd.DateOffset(months=6)

at_risk = at_risk[
    (at_risk['Expiration Date'] >= today) &
    (at_risk['Expiration Date'] <= cutoff)
].reset_index(drop=True)

print(f'Certifications expiring within 6 months: {len(at_risk)}')
at_risk

Certifications expiring within 6 months: 81


,Name,CorpEmail,Certification,Expiration Date
0,Luke Gordon,luke.gordon@contoso.com,Microsoft Certified: Azure AI Engineer Associate,2026-05-25
1,Gavin Kay,gavin.kay@contoso.com,Microsoft Certified: Agentic AI Business Solut...,2026-05-27
2,Toby Heath,toby.heath@contoso.com,Microsoft Certified: Agentic AI Business Solut...,2026-05-29
3,Mathew Howe,mathew.howe@contoso.com,Microsoft Certified: Azure AI Engineer Associate,2026-05-29
4,Zoe Murray,zoe.murray@contoso.com,Microsoft Certified: Agentic AI Business Solut...,2026-05-29
...,...,...,...,...
76,Joanne Jones,joanne.jones@contoso.com,Microsoft Certified: Azure AI Engineer Associate,2026-11-03
77,Melanie Morris,melanie.morris@contoso.com,GitHub Copilot,2026-11-06
78,Pauline Roberts,pauline.roberts@contoso.com,GitHub Copilot,2026-11-09
79,Irene Sinclair,irene.sinclair@contoso.com,Microsoft Certified: Azure AI Engineer Associate,2026-11-11
